# Minority Neuron Identification
Loads a trained SAE and identifies minority neurons for a given prompt. Before running this, ensure that the diffusion activations are hooked  and saved for the prompt since it is required during the inference.

In [ ]:
import sys, os
sys.path.insert(0, 'msae')

import torch
import numpy as np
import matplotlib.pyplot as plt
from argparse import Namespace
from PIL import Image
from datasets import Dataset
from sklearn.metrics.pairwise import cosine_distances

from config import get_default_cfg
from sae import GlobalBatchTopKMatryoshkaSAE
from get_clip_embeddings import get_clip_embeddings
from raigen import (load_image_dataset, load_sae_activations,
                    compute_dataset_centroid, compute_weighted_centroid_neuron,
                    prune_redundant_neurons, get_top_activating_images,
                    overlay_heatmap_on_image)

# ── Edit these ───────────────────────────────────────────────────────────────
PROMPT   = 'Doctor'
SAE_CKPT_DIR = f'raigen_checkpoints/prof/{PROMPT}'
IMAGE_DIR    = f'dataset_sdxl/prof/{PROMPT}/stable-diffusion-xl-base-1.0'
ACT_DIR      = f'activations_sdxl/prof/{PROMPT}/stable-diffusion-xl-base-1.0'
GROUP_SIZES  = [2048, 18432]
# ─────────────────────────────────────────────────────────────────────────────

In [2]:
cfg = Namespace(**get_default_cfg())
cfg.group_sizes  = GROUP_SIZES
cfg.dict_size    = sum(GROUP_SIZES)
cfg.device       = 'cuda:0' if torch.cuda.is_available() else 'cpu'
cfg.image_dir    = IMAGE_DIR
cfg.dataset_path = ACT_DIR
cfg.hook_point   = 'unet.mid_block'

sae = GlobalBatchTopKMatryoshkaSAE(cfg)
state_dict = torch.load(f'{SAE_CKPT_DIR}/sae.pt', map_location='cpu', weights_only=True)
sae.load_state_dict(state_dict)
sae.training = False
sae = sae.half()
print(f'SAE loaded | W_enc: {tuple(sae.W_enc.shape)} | device: {cfg.device}')

In [3]:
# Load data using raigen.py functions
sd_act_dataset = Dataset.load_from_disk(os.path.join(ACT_DIR, cfg.hook_point), keep_in_memory=False)
image_dataset  = load_image_dataset(IMAGE_DIR)

steps    = sd_act_dataset.num_rows // len(image_dataset)
timestep = steps - 1
print(f'Images: {len(image_dataset)} | Steps: {steps} | Timestep: {timestep}')

sae_activations = load_sae_activations(cfg, sae, sd_act_dataset, steps, timestep)
print(f'SAE activations: {tuple(sae_activations.shape)}')

In [ ]:
GROUP_START, GROUP_END = 0, GROUP_SIZES[0]
latents = sae_activations[:, GROUP_START:GROUP_END].numpy().astype(np.float32)

clip_path = os.path.join(IMAGE_DIR, 'clip_embed.pt')
if os.path.exists(clip_path):
    print(f'Loading existing CLIP embeddings from {clip_path}')
    embeddings = np.asarray(torch.load(clip_path, map_location='cpu'))
else:
    print(f'CLIP embeddings not found at {clip_path}, generating now...')
    embeddings = get_clip_embeddings(image_dataset, batch_size=32)
    torch.save(embeddings, clip_path)
    embeddings = np.asarray(embeddings)
    print(f'Saved CLIP embeddings to {clip_path}')

embeddings_norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

dataset_centroid  = compute_dataset_centroid(embeddings_norm)
dataset_centroid  = dataset_centroid / np.linalg.norm(dataset_centroid)
weighted_centroid = compute_weighted_centroid_neuron(embeddings_norm, latents)
weighted_centroid = weighted_centroid / (np.linalg.norm(weighted_centroid, axis=1, keepdims=True) + 1e-8)

distances  = cosine_distances(weighted_centroid, dataset_centroid.reshape(1, -1)).squeeze()
frequency  = (latents > 0.0).mean(axis=0)

active_mask       = frequency > 0
active_idx        = np.nonzero(active_mask)[0]
frequency         = frequency[active_mask]
distances         = distances[active_mask]
weighted_centroid = weighted_centroid[active_mask]

distances      = (distances - distances.min()) / (distances.max() - distances.min())
frequency      = (frequency - frequency.min()) / (frequency.max() - frequency.min())
minority_score = (1 - frequency) * distances

pruned_active    = prune_redundant_neurons(weighted_centroid, minority_score)
minority_neurons = active_idx[pruned_active]
minority_scores  = minority_score[pruned_active]
sorted_idx       = np.argsort(-minority_scores)
minority_neurons = minority_neurons[sorted_idx]
minority_scores  = minority_scores[sorted_idx]

print(f'Found {len(minority_neurons)} minority neurons')

In [ ]:
# Visualise : top-row originals, bottom-row heatmap overlays
SHOW_N = 15
TOP_K  = 5

def get_sae_heatmap_notebook(sae, sd_act_dataset, top_image_idxs, neuron, timestep, steps, image_size=256):
    all_indices = [(idx.item() * steps) + timestep for idx in top_image_idxs]
    selected = sd_act_dataset.select(all_indices)
    selected.set_format(type='torch', columns=['activations'])

    activations_col = selected['activations']
    if isinstance(activations_col, torch.Tensor):
        activations = activations_col
    else:
        activations = torch.stack([torch.as_tensor(a) for a in activations_col], dim=0)

    activations = activations.to(sae.W_dec.device, dtype=sae.W_enc.dtype)
    spatial_dim = int(activations.shape[1] ** 0.5)

    with torch.no_grad():
        sae.eval()
        feats = sae.encode(activations)
        heatmap = feats[..., neuron].reshape(len(all_indices), spatial_dim, spatial_dim)
        heatmap = torch.nn.functional.interpolate(
            heatmap.unsqueeze(1).float(),
            size=(image_size, image_size),
            mode='bilinear',
            align_corners=False,
        ).squeeze(1)
    return heatmap.cpu()

fig, axes = plt.subplots(SHOW_N * 2, TOP_K, figsize=(TOP_K * 2, SHOW_N * 4))

for row_i, neuron in enumerate(minority_neurons[:SHOW_N]):
    global_neuron = int(neuron) + GROUP_START
    acts     = sae_activations[:, global_neuron]
    top_idxs, _ = get_top_activating_images(acts, top_k=TOP_K)
    top_images   = [Image.open(image_dataset[i.item()]).convert('RGB') for i in top_idxs]
    heatmaps     = get_sae_heatmap_notebook(sae, sd_act_dataset, top_idxs, global_neuron, timestep, steps)
    overlays     = [overlay_heatmap_on_image(img, hm) for img, hm in zip(top_images, heatmaps)]

    for col_i in range(TOP_K):
        axes[row_i * 2,     col_i].imshow(top_images[col_i].resize((128, 128)))
        axes[row_i * 2,     col_i].axis('off')
        axes[row_i * 2 + 1, col_i].imshow(overlays[col_i].resize((128, 128)))
        axes[row_i * 2 + 1, col_i].axis('off')
    axes[row_i * 2, 0].set_title(
        f'neuron {global_neuron}  score={minority_scores[row_i]:.3f}',
        fontsize=7, loc='left')
plt.tight_layout()
plt.show()